# 14 · The agent in the loop: wiring Claude to propose changes

The loop has a seam: `proposer.propose(history, best_config) -> (override,
rationale)`. The real proposer is Claude (code below). With no API key here, we
run a **MockProposer** that reads the history heuristically — same interface,
same loop — so the mechanics execute end to end.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from harness.data import make_synthetic_dataset, DatasetSpec, load_split, class_names

DATA = Path("../data/bdd-tiny.lance")
if not DATA.exists():
    make_synthetic_dataset(DATA, DatasetSpec(n=3000, seed=7))
print("dataset:", DATA, "| NOTE: these chapters scale to bdd-small/full; here we")
print("demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.")

dataset: ../data/bdd-tiny.lance | NOTE: these chapters scale to bdd-small/full; here we
demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.


```python
# the real thing (needs ANTHROPIC_API_KEY):
import anthropic
class ClaudeProposer:
    def __init__(self): self.client = anthropic.Anthropic()
    def propose(self, history, best_config):
        msg = self.client.messages.create(
            model="claude-opus-4-7", max_tokens=1024,
            system=open("../program.md").read(),          # prompt-cache this
            messages=[{"role": "user", "content": format_history(history)}],
            tools=[PROPOSE_CHANGE_TOOL])
        change = parse_tool_use(msg)
        return change["override"], change["rationale"]
```

In [2]:
# a stand-in that emulates an LLM reading results.tsv and picking the next move
class MockProposer:
    PLAYBOOK = [
        ({"mining": {"enabled": True, "strategy": "fog_boost", "boost": 5.0}},
         "worst slice is foggy & rare -> upweight fog 5x"),
        ({"mining": {"enabled": True, "strategy": "fog_knn", "boost": 6.0, "k": 8}},
         "expand fog set via LanceDB neighbours"),
        ({"model": {"width": 32}}, "more capacity for the weak fog cue"),
        ({"train": {"weight_decay": 1e-4}}, "denoise high-variance fog frames"),
    ]
    def __init__(self): self.i = 0
    def propose(self, history, best_config):
        # a real agent would reason over `history`; we step the playbook
        move = self.PLAYBOOK[min(self.i, len(self.PLAYBOOK) - 1)]; self.i += 1
        return move

from harness.loop import run_loop
out = run_loop(iters=4, dataset_path=str(DATA), results_path="results.tsv",
               proposer=MockProposer(), verbose=True)
print("\nbest score:", round(out["best_score"], 4))

[iter 0] baseline score=0.883 worst=0.644(foggy)


[iter 1] KEEP score=0.972 (best=0.972) worst=0.733(foggy) :: worst slice is foggy & rare -> upweight fog 5x


[iter 2] KEEP score=0.975 (best=0.975) worst=0.733(foggy) :: expand fog set via LanceDB neighbours


[iter 3] KEEP score=1.018 (best=1.018) worst=0.778(foggy) :: more capacity for the weak fog cue


[iter 4] KEEP score=1.019 (best=1.019) worst=0.778(foggy) :: denoise high-variance fog frames

best score: 1.019


The LLM **proposes**; the **harness decides**. Claude never reports the score —
`evaluator.py` does and `loop.py` does the keep/revert. Swap `MockProposer` for
`ClaudeProposer` and set `ANTHROPIC_API_KEY` to put a real model in the seat.